# Data audit: BATADAL files

This notebook checks the three BATADAL files before any models are built. It looks at row counts, column names, data types, number precision, time order, empty cells, value ranges, constant columns, negative values, and labels. The goal is to find problems early so we can decide how to handle them.

Nothing in the data files is changed. The only fix (removing spaces from the Dataset 2 column names) happens in memory.

Files used (all in `data/`):

| Name in these notes | File | Role | Variable in the code |
|---|---|---|---|
| Dataset 1 | `Training dataset 1.csv` | Normal training data (no attacks) | `dfT1` |
| Dataset 2 | `Training dataset 2.csv` | Development data (partly labeled) | `dfT2` |
| Test set | `BATADAL_test_dataset.csv` | Final test data (no labels) | `dfTe` |

The notebook reads the files with relative paths (`../data/`), so it runs from the `notebooks` folder after the repo is cloned.

In the tables printed by the code, `T1`, `T2`, and `Te` also mean Dataset 1, Dataset 2, and Test set.

The issues and decisions we need to make are written up in `docs/audit.md`.

## 1. Load the files

Load each file and print its shape and first rows. This checks that the files open and shows the row and column counts.

Result: Dataset 1 has 8,761 rows and 45 columns. Dataset 2 has 4,177 rows and 45 columns. The Test set has 2,089 rows and 44 columns.

In [1]:
import pandas as pd

dfT1 = pd.read_csv("../data/Training dataset 1.csv")

print(dfT1.shape)
print(dfT1.head())


(8761, 45)
      DATETIME      L_T1      L_T2      L_T3      L_T4      L_T5      L_T6  \
0  06/01/14 00  0.509730  2.049003  3.191145  2.792634  2.656091  5.316831   
1  06/01/14 01  0.412580  2.009072  3.642565  2.831673  3.126387  5.494855   
2  06/01/14 02  0.320112  1.986093  4.140192  3.256733  3.574601  5.500000   
3  06/01/14 03  0.332879  2.009203  4.673478  3.744497  3.952379  5.500000   
4  06/01/14 04  0.483496  2.089049  5.237937  4.409456  3.504676  5.500000   

       L_T7      F_PU1  S_PU1  ...     P_J256     P_J289     P_J415  \
0  1.562321  98.998444      1  ...  87.605774  26.495605  84.206619   
1  1.852043  99.095901      1  ...  89.448341  26.487326  85.900085   
2  2.246126  98.420959      1  ...  91.056114  26.487364  86.582474   
3  3.203573  97.575172      1  ...  92.594353  26.575815  88.020546   
4  4.439714  97.351059      1  ...  94.473099  26.723457  90.422462   

      P_J302     P_J306     P_J307     P_J317      P_J14     P_J422  ATT_FLAG  
0  18.901676 

In [2]:
dfT2 = pd.read_csv("../data/Training dataset 2.csv")

print(dfT2.shape)
print(dfT2.head(5))

(4177, 45)
      DATETIME   L_T1   L_T2   L_T3   L_T4   L_T5   L_T6   L_T7   F_PU1  \
0  04/07/16 00   2.44   5.24   3.19   4.10   2.86   5.50   4.39   93.63   
1  04/07/16 01   2.66   4.53   3.20   4.18   3.29   5.44   4.53   89.41   
2  04/07/16 02   3.11   3.66   3.66   4.21   3.87   5.15   3.22   89.88   
3  04/07/16 03   3.62   3.04   4.17   4.04   3.56   4.98   2.40   88.10   
4  04/07/16 04   4.08   2.68   4.73   3.20   3.11   5.39   3.46   87.01   

    S_PU1  ...   P_J256   P_J289   P_J415   P_J302   P_J306   P_J307   P_J317  \
0     1.0  ...    70.00    28.22    85.87    21.69    82.72    21.58    71.99   
1     1.0  ...    87.73    24.45    84.87    29.81    86.62    29.81    59.76   
2     1.0  ...    89.29    23.90    87.11    29.85    87.64    29.85    58.50   
3     1.0  ...    91.98    27.10    68.75    31.60    64.25    31.47    72.30   
4     1.0  ...    92.11    26.76    68.74    32.30    64.23    32.17    72.53   

    P_J14   P_J422   ATT_FLAG  
0   39.33    29.64 

In [3]:
dfTe = pd.read_csv("../data/BATADAL_test_dataset.csv")

print(dfTe.shape)
print(dfTe.head(5))

(2089, 44)
      DATETIME  L_T1  L_T2  L_T3  L_T4  L_T5  L_T6  L_T7  F_PU1  S_PU1  ...  \
0  04/01/17 00  0.73  2.27  4.00  3.26  3.87  5.50  4.28  98.93      1  ...   
1  04/01/17 01  0.69  2.25  4.53  3.26  3.84  5.50  4.78  97.95      1  ...   
2  04/01/17 02  0.90  2.31  5.03  3.41  3.32  5.16  3.22  96.82      1  ...   
3  04/01/17 03  1.11  2.54  5.16  3.97  2.82  5.01  2.54  96.76      1  ...   
4  04/01/17 04  1.27  2.99  4.94  4.27  2.35  5.38  3.41  94.77      1  ...   

   P_J300  P_J256  P_J289  P_J415  P_J302  P_J306  P_J307  P_J317  P_J14  \
0   26.68   90.54   26.74   84.52   19.43   83.27   19.33   71.33  29.61   
1   26.67   90.43   26.73   85.04   25.97   64.22   25.86   73.79  29.63   
2   26.83   91.72   26.89   87.16   29.18   63.81   29.18   59.05  29.80   
3   28.27   76.12   28.23   88.83   26.53   63.42   26.41   70.92  30.80   
4   31.32   75.68   31.32   69.55   27.46   63.43   27.34   70.88  33.61   

   P_J422  
0   28.71  
1   28.73  
2   28.90  
3   29.90

## 2. Column names

Compare the column names in the three files. The models need the same features, with the same names, in every file.

Result: Every name in Dataset 2 except DATETIME had a leading space. After `.strip()`, all three files have the same 43 feature columns. The only extra column is ATT_FLAG in Dataset 1 and Dataset 2.

In [4]:
print(len(dfT1.columns), len(dfT2.columns), len(dfTe.columns))

print(list(dfT1.columns))
print(list(dfT2.columns))
print(list(dfTe.columns))

45 45 44
['DATETIME', 'L_T1', 'L_T2', 'L_T3', 'L_T4', 'L_T5', 'L_T6', 'L_T7', 'F_PU1', 'S_PU1', 'F_PU2', 'S_PU2', 'F_PU3', 'S_PU3', 'F_PU4', 'S_PU4', 'F_PU5', 'S_PU5', 'F_PU6', 'S_PU6', 'F_PU7', 'S_PU7', 'F_PU8', 'S_PU8', 'F_PU9', 'S_PU9', 'F_PU10', 'S_PU10', 'F_PU11', 'S_PU11', 'F_V2', 'S_V2', 'P_J280', 'P_J269', 'P_J300', 'P_J256', 'P_J289', 'P_J415', 'P_J302', 'P_J306', 'P_J307', 'P_J317', 'P_J14', 'P_J422', 'ATT_FLAG']
['DATETIME', ' L_T1', ' L_T2', ' L_T3', ' L_T4', ' L_T5', ' L_T6', ' L_T7', ' F_PU1', ' S_PU1', ' F_PU2', ' S_PU2', ' F_PU3', ' S_PU3', ' F_PU4', ' S_PU4', ' F_PU5', ' S_PU5', ' F_PU6', ' S_PU6', ' F_PU7', ' S_PU7', ' F_PU8', ' S_PU8', ' F_PU9', ' S_PU9', ' F_PU10', ' S_PU10', ' F_PU11', ' S_PU11', ' F_V2', ' S_V2', ' P_J280', ' P_J269', ' P_J300', ' P_J256', ' P_J289', ' P_J415', ' P_J302', ' P_J306', ' P_J307', ' P_J317', ' P_J14', ' P_J422', ' ATT_FLAG']
['DATETIME', 'L_T1', 'L_T2', 'L_T3', 'L_T4', 'L_T5', 'L_T6', 'L_T7', 'F_PU1', 'S_PU1', 'F_PU2', 'S_PU2', 'F_PU3

Check if Dataset 1 and Dataset 2 have the same column names in the same order.

In [5]:
print(list(dfT1.columns) == list(dfT2.columns))

False


In [6]:
print(set(dfT2.columns) - set(dfT1.columns))

{' S_PU11', ' L_T6', ' S_PU3', ' F_PU7', ' F_PU4', ' S_PU7', ' P_J256', ' P_J306', ' S_V2', ' F_PU5', ' P_J300', ' S_PU8', ' L_T2', ' ATT_FLAG', ' P_J14', ' S_PU2', ' F_PU11', ' S_PU6', ' P_J415', ' S_PU9', ' P_J422', ' F_V2', ' F_PU3', ' S_PU10', ' S_PU4', ' L_T5', ' F_PU1', ' L_T7', ' F_PU6', ' P_J280', ' P_J289', ' F_PU9', ' S_PU1', ' L_T1', ' P_J317', ' F_PU10', ' P_J269', ' F_PU8', ' S_PU5', ' L_T3', ' P_J302', ' L_T4', ' P_J307', ' F_PU2'}


The names do not match. In Dataset 2 every column name except DATETIME starts with a space, for example `' L_T1'` instead of `'L_T1'`. The raw header line in section 3 shows why: the Dataset 2 file has a space after each comma.

The next cell removes the spaces with `.str.strip()`. Dataset 1 and the Test set are stripped too, just in case. This only changes the names in memory, not the files.

In [7]:
dfT1.columns = dfT1.columns.str.strip() # just in case
dfT2.columns = dfT2.columns.str.strip()
dfTe.columns = dfTe.columns.str.strip() # just in case

In [8]:
print(list(dfT1.columns) == list(dfT2.columns))

True


After stripping, Dataset 1 and Dataset 2 have the same column names in the same order. Next, check which columns are in the training files but not in the Test set.

In [9]:
print(set(dfT1.columns) - set(dfTe.columns))
print(set(dfT2.columns) - set(dfTe.columns))

{'ATT_FLAG'}
{'ATT_FLAG'}


The only difference is ATT_FLAG, the label column. The Test set has no labels, which is expected.

So Dataset 1 and Dataset 2 have 45 columns: DATETIME, 43 features, and ATT_FLAG. The Test set has 44 columns. The 43 features are:
- 31 continuous: L_T1 to L_T7 (tank levels), 12 F_ columns (flows: F_PU1 to F_PU11 and F_V2), and 12 P_ columns (pressures).
- 12 on/off: S_PU1 to S_PU11 and S_V2 (pump and valve status).

## 3. Data types

Compare the data type of each column in the three files. A type mismatch can point to bad values or to a format difference between files.

Result: DATETIME is text in all three files, and every other column is numeric. Some columns are int in one file and float in another. This comes from how the numbers are written in the files, not from bad values.

In [10]:
side = pd.DataFrame({
    "T1": dfT1.dtypes,
    "T2": dfT2.dtypes,
    "Te": dfTe.dtypes,
}).reindex(dfT1.columns)

print(side)

counts = pd.DataFrame({
    "T1": dfT1.dtypes.value_counts(),
    "T2": dfT2.dtypes.value_counts(),
    "Te": dfTe.dtypes.value_counts(),
})
print(counts)

               T1       T2       Te
DATETIME   object   object   object
L_T1      float64  float64  float64
L_T2      float64  float64  float64
L_T3      float64  float64  float64
L_T4      float64  float64  float64
L_T5      float64  float64  float64
L_T6      float64  float64  float64
L_T7      float64  float64  float64
F_PU1     float64  float64  float64
S_PU1       int64  float64    int64
F_PU2     float64  float64  float64
S_PU2       int64  float64    int64
F_PU3       int64  float64  float64
S_PU3       int64  float64    int64
F_PU4     float64  float64  float64
S_PU4       int64  float64    int64
F_PU5       int64  float64    int64
S_PU5       int64  float64    int64
F_PU6     float64  float64  float64
S_PU6       int64  float64    int64
F_PU7     float64  float64  float64
S_PU7       int64  float64    int64
F_PU8     float64  float64  float64
S_PU8       int64  float64    int64
F_PU9       int64  float64    int64
S_PU9       int64  float64    int64
F_PU10    float64  float64  

DATETIME is read as text (`object`). It is converted to real dates in section 6 for the time checks.

All other columns are numeric. Dataset 2 has 43 float columns and only 1 int column (ATT_FLAG). Dataset 1 has 28 float and 16 int columns. The Test set has 28 float and 15 int columns. The next three cells list the columns where the types differ.

In [11]:
cmp = pd.DataFrame({"T1": dfT1.dtypes, "T2": dfT2.dtypes})
print(cmp[cmp["T1"] != cmp["T2"]])

           T1       T2
S_PU1   int64  float64
S_PU2   int64  float64
F_PU3   int64  float64
S_PU3   int64  float64
S_PU4   int64  float64
F_PU5   int64  float64
S_PU5   int64  float64
S_PU6   int64  float64
S_PU7   int64  float64
S_PU8   int64  float64
F_PU9   int64  float64
S_PU9   int64  float64
S_PU10  int64  float64
S_PU11  int64  float64
S_V2    int64  float64


In [12]:
cmp2 = pd.DataFrame({"T1": dfT1.drop(columns="ATT_FLAG").dtypes, "Te": dfTe.dtypes})
print(cmp2[cmp2["T1"] != cmp2["Te"]])

             T1       Te
F_PU3     int64  float64
F_PU11  float64    int64


In [13]:
cmp3 = pd.DataFrame({"T2": dfT2.drop(columns="ATT_FLAG").dtypes, "Te": dfTe.dtypes})
print(cmp3[cmp3["T2"] != cmp3["Te"]])

             T2     Te
S_PU1   float64  int64
S_PU2   float64  int64
S_PU3   float64  int64
S_PU4   float64  int64
F_PU5   float64  int64
S_PU5   float64  int64
S_PU6   float64  int64
S_PU7   float64  int64
S_PU8   float64  int64
F_PU9   float64  int64
S_PU9   float64  int64
S_PU10  float64  int64
F_PU11  float64  int64
S_PU11  float64  int64
S_V2    float64  int64


Columns where the types differ:
- Dataset 1 vs Dataset 2: 15 columns (all 12 S_ columns plus F_PU3, F_PU5, F_PU9). These are int in Dataset 1 and float in Dataset 2.
- Dataset 1 vs Test set: 2 columns. F_PU3 is int in Dataset 1 and float in the Test set. F_PU11 is float in Dataset 1 and int in the Test set. This fits with sections 8 and 9: F_PU3 is always 0 in Dataset 1 but has real values in the Test set, and F_PU11 is the other way around.
- Dataset 2 vs Test set: 15 columns (all 12 S_ columns plus F_PU5, F_PU9, F_PU11). These are float in Dataset 2 and int in the Test set.

The next cells check if the 15 float columns in Dataset 2 have empty cells or real decimals.

In [14]:
diff_cols = side[side["T1"] != side["T2"]].index
print(dfT2[diff_cols].isna().sum())

S_PU1     0
S_PU2     0
F_PU3     0
S_PU3     0
S_PU4     0
F_PU5     0
S_PU5     0
S_PU6     0
S_PU7     0
S_PU8     0
F_PU9     0
S_PU9     0
S_PU10    0
S_PU11    0
S_V2      0
dtype: int64


None of the 15 columns has empty cells in Dataset 2. So the float type is not caused by missing values (pandas reads an int column with empty cells as float).

In [15]:
print((dfT2[diff_cols].dropna() % 1 != 0).sum())

S_PU1     0
S_PU2     0
F_PU3     0
S_PU3     0
S_PU4     0
F_PU5     0
S_PU5     0
S_PU6     0
S_PU7     0
S_PU8     0
F_PU9     0
S_PU9     0
S_PU10    0
S_PU11    0
S_V2      0
dtype: int64


All values in these 15 columns are whole numbers. So the float type is not hiding real decimals. The next two cells print the first lines of the raw Dataset 2 and Dataset 1 files to see how the numbers are written.

In [16]:
with open("../data/Training dataset 2.csv") as f:
    print(f.readlines()[:3])

['DATETIME, L_T1, L_T2, L_T3, L_T4, L_T5, L_T6, L_T7, F_PU1, S_PU1, F_PU2, S_PU2, F_PU3, S_PU3, F_PU4, S_PU4, F_PU5, S_PU5, F_PU6, S_PU6, F_PU7, S_PU7, F_PU8, S_PU8, F_PU9, S_PU9, F_PU10, S_PU10, F_PU11, S_PU11, F_V2, S_V2, P_J280, P_J269, P_J300, P_J256, P_J289, P_J415, P_J302, P_J306, P_J307, P_J317, P_J14, P_J422, ATT_FLAG\n', '04/07/16 00, 2.44, 5.24, 3.19, 4.10, 2.86, 5.50, 4.39, 93.63, 1.00, 93.65, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 49.84, 1.00, 35.54, 1.00, 0.00, 0.00, 29.87, 1.00, 0.00, 0.00, 0.00, 0.00, 2.98, 36.38, 28.26, 70.00, 28.22, 85.87, 21.69, 82.72, 21.58, 71.99, 39.33, 29.64, -999\n', '04/07/16 01, 2.66, 4.53, 3.20, 4.18, 3.29, 5.44, 4.53, 89.41, 1.00, 89.43, 1.00, 0.00, 0.00, 34.26, 1.00, 0.00, 0.00, 0.00, 0.00, 48.38, 1.00, 37.83, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 2.98, 38.60, 24.39, 87.73, 24.45, 84.87, 29.81, 86.62, 29.81, 59.76, 42.17, 26.15, -999\n']


In [17]:
with open("../data/Training dataset 1.csv") as f:
    print(f.readlines()[:3])

['DATETIME,L_T1,L_T2,L_T3,L_T4,L_T5,L_T6,L_T7,F_PU1,S_PU1,F_PU2,S_PU2,F_PU3,S_PU3,F_PU4,S_PU4,F_PU5,S_PU5,F_PU6,S_PU6,F_PU7,S_PU7,F_PU8,S_PU8,F_PU9,S_PU9,F_PU10,S_PU10,F_PU11,S_PU11,F_V2,S_V2,P_J280,P_J269,P_J300,P_J256,P_J289,P_J415,P_J302,P_J306,P_J307,P_J317,P_J14,P_J422,ATT_FLAG\n', '06/01/14 00,0.509729922,2.049002886,3.191145182,2.792634249,2.656090975,5.316830635,1.562320828,98.9984436,1,99.01815033,1,0,0,35.53668594,1,0,0,0,0,49.80932999,1,34.350811,1,0,0,30.51271248,1,0,0,81.60540771,1,2.974052906,33.50603104,26.4264946,87.60577393,26.49560547,84.20661926,18.90167618,81.98373413,18.79177666,67.12560272,29.38747025,28.48747063,0\n', '06/01/14 01,0.41258049,2.009071827,3.642565489,2.831672668,3.126387358,5.494855404,1.852042913,99.09590149,1,99.11563873,1,0,0,34.45491409,1,0,0,0,0,48.92192459,1,34.21959305,1,0,0,30.4804554,1,0,0,81.18103027,1,2.974005461,33.45324707,26.42296219,89.44834137,26.48732567,85.90008545,18.84932899,82.15058899,18.7396431,67.17869568,29.35425568,28.4542

The raw lines show the reason. Dataset 2 writes these values as `1.00` and `0.00`, so pandas reads them as float. Dataset 1 writes them as `1` and `0`, so pandas reads them as int. The values are the same, only the format is different.

The Dataset 2 header also has a space after each comma, which is where the leading spaces in the column names came from.

Raw lines from the Test set were not printed. Its S_ columns are read as int, which suggests the Test set file also writes them as `1` and `0`.

## 4. Number precision

Read the files as text and count the most decimal places in any sensor value. If the files are rounded differently, some small differences between them are only about rounding.

Result: Dataset 1 has up to 9 decimal places. Dataset 2 and the Test set have at most 2.

In [18]:
raw1 = pd.read_csv("../data/Training dataset 1.csv", dtype=str)
raw2 = pd.read_csv("../data/Training dataset 2.csv", dtype=str)
raw3 = pd.read_csv("../data/BATADAL_test_dataset.csv", dtype=str)

for r in (raw1, raw2, raw3):
    r.columns = r.columns.str.strip()

def max_decimals(raw):
    sensors = raw.drop(columns=["DATETIME", "ATT_FLAG"], errors="ignore")
    return sensors.apply(
        lambda c: c.str.strip().str.split(".").str[1].fillna("").str.len().max()
    )

print(max_decimals(raw1).max(), max_decimals(raw2).max(), max_decimals(raw3).max())

9 2 2


Max decimal places: Dataset 1 = 9, Dataset 2 = 2, Test set = 2. So the values in Dataset 1 are rounded differently from Dataset 2 and the Test set. This is listed as an open question for us.

## 5. On/off columns

Check that the 12 S_ columns contain only 0 and 1. These columns are pump and valve status, so any other value would be an error.

Result: all three files have 12 S_ columns, and every value in them is 0 or 1.

In [19]:
s1 = dfT1.filter(like="S_")
s2 = dfT2.filter(like="S_")
se = dfTe.filter(like="S_")
print(s1.shape[1], s2.shape[1], se.shape[1])

12 12 12


In [20]:
print((~s1.isin([0, 1])).sum().sum())
print((~s2.isin([0, 1])).sum().sum())
print((~se.isin([0, 1])).sum().sum())

0
0
0


## 6. Time order and gaps

Convert DATETIME to real dates and check the start, end, order, repeated hours, and gaps. The format is day/month/year hour, for example `06/01/14 00` is 6 January 2014, 00:00. The time models (GRU, Transformer) need hourly rows in order with no gaps.

Result: all three files are hourly, in order, with no repeated hours and no missing hours.

In [21]:
t1 = pd.to_datetime(dfT1["DATETIME"], format="%d/%m/%y %H")
print(t1.min(), t1.max())

print(t1.is_monotonic_increasing)
print(t1.duplicated().sum())

print(t1.diff().value_counts())

2014-01-06 00:00:00 2015-01-06 00:00:00
True
0
DATETIME
0 days 01:00:00    8760
Name: count, dtype: int64


In [22]:
t2 = pd.to_datetime(dfT2["DATETIME"], format="%d/%m/%y %H")
print(t2.min(), t2.max())

print(t2.is_monotonic_increasing)
print(t2.duplicated().sum())

print(t2.diff().value_counts())

2016-07-04 00:00:00 2016-12-25 00:00:00
True
0
DATETIME
0 days 01:00:00    4176
Name: count, dtype: int64


In [23]:
te = pd.to_datetime(dfTe["DATETIME"], format="%d/%m/%y %H")
print(te.min(), te.max())

print(te.is_monotonic_increasing)
print(te.duplicated().sum())

print(te.diff().value_counts())

2017-01-04 00:00:00 2017-04-01 00:00:00
True
0
DATETIME
0 days 01:00:00    2088
Name: count, dtype: int64


Results:

| File | Start | End | In order | Repeated hours | Time between rows |
|---|---|---|---|---|---|
| Dataset 1 | 2014-01-06 00:00 | 2015-01-06 00:00 | yes | 0 | 1 hour for all 8,760 steps |
| Dataset 2 | 2016-07-04 00:00 | 2016-12-25 00:00 | yes | 0 | 1 hour for all 4,176 steps |
| Test set | 2017-01-04 00:00 | 2017-04-01 00:00 | yes | 0 | 1 hour for all 2,088 steps |

The converted dates are saved in `t1`, `t2`, and `te` and are used later in the notebook. The DATETIME column inside the data frames is still text.

## 7. Empty cells

Count the empty (NaN) cells in each file.

Result: there are no empty cells in any of the three files.

In [24]:
print(dfT1.isna().sum().sum())
print(dfT2.isna().sum().sum())
print(dfTe.isna().sum().sum())

0
0
0


0 empty cells in Dataset 1, Dataset 2, and the Test set. No filling is needed.

## 8. Value ranges

Print summary stats (count, mean, std, min, quartiles, max) for every column in each file. This shows odd values, columns that never change, and differences between the files. The findings are in the note after the three tables.

In [25]:
print(dfT1.describe().T)

           count        mean        std        min        25%        50%  \
L_T1      8761.0    2.676905   1.202940   0.320112   1.650647   2.798855   
L_T2      8761.0    3.285748   1.468621   0.293876   2.131776   3.498371   
L_T3      8761.0    4.201820   0.678465   2.882847   3.623394   4.240439   
L_T4      8761.0    3.568129   0.539656   2.002133   3.139640   3.574840   
L_T5      8761.0    2.748205   0.735162   1.293241   2.123508   2.758198   
L_T6      8761.0    5.370437   0.177459   4.824605   5.248147   5.500000   
L_T7      8761.0    3.302672   0.828849   1.054194   2.631062   3.252838   
F_PU1     8761.0  100.926032  10.048618  85.495201  94.342598  96.729073   
S_PU1     8761.0    1.000000   0.000000   1.000000   1.000000   1.000000   
F_PU2     8761.0   69.464357  42.239742   0.000000   0.000000  94.075195   
S_PU2     8761.0    0.730624   0.443661   0.000000   0.000000   1.000000   
F_PU3     8761.0    0.000000   0.000000   0.000000   0.000000   0.000000   
S_PU3     87

In [26]:
print(dfT2.describe().T)

           count        mean         std     min     25%     50%     75%  \
L_T1      4177.0    2.707446    1.209627    0.32    1.65    2.82    3.78   
L_T2      4177.0    3.287965    1.470462    0.31    2.15    3.49    4.58   
L_T3      4177.0    4.204496    0.681352    2.88    3.62    4.23    4.80   
L_T4      4177.0    3.556232    0.544905    1.92    3.13    3.55    4.00   
L_T5      4177.0    2.748377    0.735800    1.28    2.13    2.75    3.37   
L_T6      4177.0    5.368429    0.177560    4.77    5.24    5.50    5.50   
L_T7      4177.0    3.323352    0.838134    0.43    2.66    3.28    4.04   
F_PU1     4177.0  100.839069   10.217831   38.92   94.17   96.65  112.14   
S_PU1     4177.0    1.000000    0.000000    1.00    1.00    1.00    1.00   
F_PU2     4177.0   69.385904   42.180931    0.00    0.00   93.86   96.51   
S_PU2     4177.0    0.730907    0.443541    0.00    0.00    1.00    1.00   
F_PU3     4177.0    0.000000    0.000000    0.00    0.00    0.00    0.00   
S_PU3     41

In [27]:
print(dfTe.describe().T)

         count        mean        std    min    25%    50%     75%     max
L_T1    2089.0    2.779449   1.211617   0.52   1.77   2.94    3.83    6.34
L_T2    2089.0    3.353676   1.441290   0.31   2.25   3.59    4.57    5.71
L_T3    2089.0    4.198928   0.677163   2.88   3.61   4.23    4.80    5.43
L_T4    2089.0    3.570369   0.543436   2.02   3.14   3.58    4.01    4.66
L_T5    2089.0    2.753021   0.736856   1.33   2.13   2.76    3.39    4.15
L_T6    2089.0    5.374840   0.172273   4.82   5.26   5.50    5.50    5.50
L_T7    2089.0    3.294955   0.818310   1.24   2.61   3.20    3.99    4.98
F_PU1   2089.0  100.263605  12.207435   0.00  94.09  96.52  111.99  122.44
S_PU1   2089.0    0.995213   0.069039   0.00   1.00   1.00    1.00    1.00
F_PU2   2089.0   67.099746  43.313361   0.00   0.00  93.72   96.37   99.97
S_PU2   2089.0    0.706558   0.455448   0.00   0.00   1.00    1.00    1.00
F_PU3   2089.0    2.749751  16.072286   0.00   0.00   0.00    0.00  119.56
S_PU3   2089.0    0.02872

What the tables show:
- **L_T6 stops at 5.5** in all three files. The median, 75th percentile, and max are all 5.5. It looks like an upper limit on that tank. It was not treated as an error.
- **Pumps 6 and 11 are almost always off** in Dataset 1 and Dataset 2. F_PU6, S_PU6, F_PU11, and S_PU11 have a 75th percentile of 0 but a max above 0. From the mean of the status columns, pump 6 is on for 16 hours in Dataset 1 and 87 hours in Dataset 2. Pump 11 is on for 4 hours in Dataset 1 and 32 hours in Dataset 2.
- **Some columns never change** (min = max). These are listed in section 9.
- **Negative values:** Dataset 1 and the Test set have no negative minimum. Dataset 2 has one: F_PU7 = -0.88. This is checked in section 10.
- **The Test set differs from the training files in a few places.** F_PU3 and S_PU3 are not always 0 (F_PU3 reaches 119.56). S_PU1 is 0 in some rows (mean 0.995, so about 10 rows). A few sensors go above the training max, for example L_T1 reaches 6.34 (max 4.87 in Dataset 1 and 5.73 in Dataset 2) and P_J415 reaches 122.93 (max 105.95 and 107.95).

The Test set was only looked at for summary values like these. No model choice or threshold is based on it.

## 9. Columns that never change

List the columns that have only one unique value in each file. A column that never changes gives the model no information, and MinMaxScaler has no range to scale it with.

Result: the same 7 features never change in Dataset 1 and Dataset 2.

In [28]:
unique = pd.DataFrame({
    "T1": dfT1.nunique()[dfT1.nunique() == 1],
    "T2": dfT2.nunique()[dfT2.nunique() == 1],
    "Te": dfTe.nunique()[dfTe.nunique() == 1],
}).dropna(how="all")

print(unique)

           T1   T2   Te
ATT_FLAG  1.0  NaN  NaN
F_PU11    NaN  NaN  1.0
F_PU3     1.0  1.0  NaN
F_PU5     1.0  1.0  1.0
F_PU9     1.0  1.0  1.0
S_PU1     1.0  1.0  NaN
S_PU11    NaN  NaN  1.0
S_PU3     1.0  1.0  NaN
S_PU5     1.0  1.0  1.0
S_PU9     1.0  1.0  1.0


A 1.0 means the column has only one value in that file. NaN means the column changes in that file.
- Dataset 1 and Dataset 2: S_PU1 is always 1. F_PU3, S_PU3, F_PU5, S_PU5, F_PU9, and S_PU9 are always 0 (values from section 8). So pump 1 is always on, and pumps 3, 5, and 9 are always off.
- Dataset 1: ATT_FLAG is also constant. It is always 0.
- Test set: F_PU5, S_PU5, F_PU9, S_PU9, F_PU11, and S_PU11 never change (all 0). F_PU3 and S_PU3 are **not** constant in the Test set, so pump 3 runs in the Test set even though it never runs in Dataset 1 or Dataset 2. S_PU1 is not constant in the Test set either.

Whether to drop or keep the 7 constant features is a decision we need to make.

## 10. Negative values

Count the negative values in every numeric column. Flows, tank levels, and pressures should not be negative. The cells below check Dataset 1, then the Test set, then Dataset 2.

Result: Dataset 1 and the Test set have no negative values. Dataset 2 has one, in F_PU7.

In [29]:
num = dfT1.select_dtypes("number").drop(columns="ATT_FLAG", errors="ignore")
neg_counts = (num < 0).sum()
print(neg_counts[neg_counts > 0])

Series([], dtype: int64)


In [30]:
num = dfTe.select_dtypes("number").drop(columns="ATT_FLAG", errors="ignore")
neg_counts = (num < 0).sum()
print(neg_counts[neg_counts > 0])

Series([], dtype: int64)


In [31]:
num = dfT2.select_dtypes("number").drop(columns="ATT_FLAG", errors="ignore")
neg_counts = (num < 0).sum()
print(neg_counts[neg_counts > 0])

F_PU7    1
dtype: int64


Dataset 2 has 1 negative value, in F_PU7. The next cell prints the two rows before and after it, and its time.

In [32]:
neg = dfT2[dfT2["F_PU7"] < 0]
i = neg.index[0]
print(dfT2.loc[i-2:i+2, ["F_PU7", "S_PU7", "ATT_FLAG"]])
print(t2[i])

      F_PU7  S_PU7  ATT_FLAG
3767   0.00    0.0         1
3768   0.00    0.0         1
3769  -0.88    1.0         1
3770   0.43    1.0         1
3771   0.00    0.0         1
2016-12-08 01:00:00


The negative value is F_PU7 = -0.88 at 2016-12-08 01:00. On that row S_PU7 is 1 (pump 7 on). On the rows around it the flow is 0 or close to 0 (0.00, 0.00, -0.88, 0.43, 0.00), and the pump switches off, off, on, on, off.

All five rows are labeled as attack (ATT_FLAG = 1). The row is inside the labeled attack from 2016-12-06 21:00 to 2016-12-09 21:00 (section 12), so the attack may be the cause. The value was left as is.

## 11. Labels (ATT_FLAG)

Count the label values in Dataset 2. The labels decide how Dataset 2 can be used for training and evaluation.

Result: 3,958 rows are -999 and 219 rows are 1. No row is 0.

In [33]:
print(dfT2["ATT_FLAG"].value_counts())

ATT_FLAG
-999    3958
 1       219
Name: count, dtype: int64


Dataset 2 labels: -999 = 3,958 rows, 1 = 219 rows, 0 = no rows. Here 1 is read as attack and -999 as unlabeled (not known). So no row in Dataset 2 is labeled as confirmed normal.

For comparison: Dataset 1 is all 0 (ATT_FLAG never changes in section 9, and its max is 0 in section 8). The Test set has no label column.

## 12. Attack periods in Dataset 2

Find each run of back-to-back hours labeled 1, and compare the runs with the attack table on the BATADAL site (saved as `docs/attacks_dataset2.png`). This shows how much of each real attack is labeled.

Result: 5 labeled runs, 219 hours in total. The site lists 7 attacks. Only attack 3 is fully labeled. Attacks 1, 4, 5, and 6 are partly labeled, and attacks 2 and 7 have no labeled hours.

![BATADAL site table of the 7 attacks in Dataset 2](../docs/attacks_dataset2.png)

In [34]:
is_att = dfT2["ATT_FLAG"] == 1
starts = t2[is_att & ~is_att.shift(1, fill_value=False)]
ends = t2[is_att & ~is_att.shift(-1, fill_value=False)]

print(len(starts))
for s, e in zip(starts, ends):
    print(s, "to", e, "->", int((e - s) / pd.Timedelta(hours=1)) + 1, "hours")

5
2016-09-14 03:00:00 to 2016-09-15 20:00:00 -> 42 hours
2016-10-09 09:00:00 to 2016-10-11 20:00:00 -> 60 hours
2016-10-30 19:00:00 to 2016-11-01 07:00:00 -> 37 hours
2016-11-27 04:00:00 to 2016-11-27 10:00:00 -> 7 hours
2016-12-06 21:00:00 to 2016-12-09 21:00:00 -> 73 hours


Labeled runs (from the cell above) next to the site's table:

| Site ID | Real start | Real end | Real hours | Labeled start | Labeled end | Labeled hours |
|---|---|---|---|---|---|---|
| 1 | 2016-09-13 23:00 | 2016-09-16 00:00 | 50 | 2016-09-14 03:00 | 2016-09-15 20:00 | 42 |
| 2 | 2016-09-26 11:00 | 2016-09-27 10:00 | 24 | none | none | 0 |
| 3 | 2016-10-09 09:00 | 2016-10-11 20:00 | 60 | 2016-10-09 09:00 | 2016-10-11 20:00 | 60 |
| 4 | 2016-10-29 19:00 | 2016-11-02 16:00 | 94 | 2016-10-30 19:00 | 2016-11-01 07:00 | 37 |
| 5 | 2016-11-26 17:00 | 2016-11-29 04:00 | 60 | 2016-11-27 04:00 | 2016-11-27 10:00 | 7 |
| 6 | 2016-12-06 07:00 | 2016-12-10 04:00 | 94 | 2016-12-06 21:00 | 2016-12-09 21:00 | 73 |
| 7 | 2016-12-14 15:00 | 2016-12-19 04:00 | 110 | none | none | 0 |
| Total | | | 492 | | | 219 |

The labeled hours match the site's "Labeled" column for every attack. Each labeled run is inside the real attack window. The labeled start is later than the real start by 4 hours (attack 1), 24 hours (attack 4), 11 hours (attack 5), and 14 hours (attack 6). For attack 3 the two start times are the same.

The next cell checks the attack 7 window directly.

In [35]:
mask = (t2 >= "2016-12-14 15:00") & (t2 <= "2016-12-19 4:00")
print(dfT2.loc[mask, "ATT_FLAG"].value_counts())

ATT_FLAG
-999    110
Name: count, dtype: int64


All 110 rows in the attack 7 window (2016-12-14 15:00 to 2016-12-19 04:00) are -999. So attack 7 is in the data, but none of it is labeled. The next cell adds up the hours in the 5 labeled runs.

In [36]:
total = 0
for s, e in zip(starts, ends):
    total += int((e - s) / pd.Timedelta(hours=1)) + 1
print(total)

219


The 5 labeled runs add up to 219 hours. This is the same as the number of rows with ATT_FLAG = 1 (section 11), so every 1 is inside one of these runs.

The real attacks on the site's table add up to 492 hours. So about 273 attack hours (492 - 219) are inside the -999 rows, if the site's list is complete. Also, every row before 2016-09-14 03:00 is -999.

## Summary

- **Rows:** Dataset 1 = 8,761, Dataset 2 = 4,177, Test set = 2,089.
- **Columns:** all three files have the same 43 features (31 continuous, 12 on/off). Dataset 1 and Dataset 2 also have ATT_FLAG.
- **Column names:** Dataset 2 had a leading space on every name except DATETIME. Fixed in memory with `.strip()`.
- **Types:** DATETIME is text. Some numeric columns are int in one file and float in another, because Dataset 2 writes `1.00` and `0.00`. The values are the same.
- **Precision:** Dataset 1 has up to 9 decimal places. Dataset 2 and the Test set have at most 2.
- **On/off columns:** only 0 and 1 in all three files.
- **Time:** all three files are hourly, in order, with no repeated or missing hours.
- **Empty cells:** none.
- **Constant columns:** 7 features never change in Dataset 1 and Dataset 2 (S_PU1 always 1; F_PU3, S_PU3, F_PU5, S_PU5, F_PU9, S_PU9 always 0).
- **L_T6** stops at 5.5 in all three files.
- **Negative value:** one, F_PU7 = -0.88 in Dataset 2, inside a labeled attack. Left as is.
- **Test set (for information only):** pump 3 runs, S_PU1 is 0 in a few rows, and some values go above the training max.
- **Labels:** Dataset 1 is all 0. Dataset 2 has 3,958 rows of -999, 219 rows of 1, and no 0. The labels cover 219 of the 492 real attack hours. Attacks 2 and 7 have no labeled hours.

The issues and decisions we need to make are in `docs/audit.md`.